# 04c — Alternate-camera ball-observation pipeline from raw data

## Problem

Possession transitions are hardest when the tactical camera loses the ball. A missing coordinate does not tell us whether the ball was occluded, travelling, carried by a player, or out of play. If we label that interval directly, we risk inventing a switch.

## Idea

Run George's ballfill system from the raw match bundle, produce a frame-aligned observation dataframe, and pass that evidence into our existing feature/scenario/anchor/transition pipeline. George reconstructs ball geometry; our resolver remains responsible for H/A/D/X.

This notebook does not require a precomputed `BAR-ATH_ball.csv`. That file is only an optional generated audit artifact.

## What George's model actually contains

The branch is not a YOLO pipeline. It contains:

1. tactical/broadcast ball and player alignment;
2. a small heatmap U-Net trained from tracker agreement, rather than hand-labelled ball images;
3. tracklet verification for candidate video detections;
4. a carrier classifier and dead-ball classifier trained on held-out/open-match data; and
5. a forward-backward carrier/dead-ball HMM that fills unresolved gaps.

The HMM estimates latent ball-path causes. It does not become our final possession decoder.

In [1]:
import os
from pathlib import Path
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'possession_lab').exists():
    ROOT = ROOT.parent
DATA_ROOT = Path(os.environ.get('MTAH_DATA_DIR', ROOT / 'data'))

# Set these explicitly when presenting or rerunning the raw-data stage.
RUN_GEORGE_VIDEO_TRAINING = False
RUN_GEORGE_OBSERVATION_STAGE = False
GEORGE_TRAINING_STEP = 'all'  # extract, train, detect, or all

from possession_lab.george_ballfill_stage import (
    train_george_video_models,
    build_our_observations_with_george,
)

print('raw data root:', DATA_ROOT)
print('George video training enabled:', RUN_GEORGE_VIDEO_TRAINING)
print('George observation stage enabled:', RUN_GEORGE_OBSERVATION_STAGE)

raw data root: /Users/luke/Desktop/barca-hack-repo-staging/data
George video training enabled: False
George observation stage enabled: False


## Train or load the observation model

Training is deliberately a separate toggle. In a normal pipeline run, previously trained model/cache files are reused. The raw match bundle remains the source; the generated cache is an internal reproducibility artifact, not a supplied label file.

The carrier/dead-ball models use George's documented external/open-match training procedure. They are supervised/model-based components for ball reconstruction, not training on the hackathon possession ground truth.

In [2]:
if RUN_GEORGE_VIDEO_TRAINING:
    train_george_video_models(data_dir=DATA_ROOT, step=GEORGE_TRAINING_STEP)
else:
    print('Training skipped: set RUN_GEORGE_VIDEO_TRAINING=True to run the raw-video training stage.')

Training skipped: set RUN_GEORGE_VIDEO_TRAINING=True to run the raw-video training stage.


## Generate George's observations and pass them downstream

The stage returns two tables:

- `george_table`: one row per tactical frame with coordinates, source, `sigma_m`, `p_video`, `p_carrier`, and `p_dead`;
- `features`: our complete frame-level feature table after George evidence has passed through temporal calculations, scenarios, anchors, and transition evidence.

The default is evidence-only. Tactical coordinates stay primary; George confidence can strengthen missing-ball evidence without manufacturing a possession switch.

In [3]:
if RUN_GEORGE_OBSERVATION_STAGE:
    features, george_table = build_our_observations_with_george(
        data_dir=DATA_ROOT,
        evidence_only=True,
        allow_model_filled=False,
        minimum_confidence=None,
    )
    print('George rows:', len(george_table))
    print('Our feature rows:', len(features))
    display(george_table.groupby('source', dropna=False).size().sort_values(ascending=False).to_frame('frames'))
else:
    print('Observation stage skipped: set RUN_GEORGE_OBSERVATION_STAGE=True after configuring the raw data bundle.')

Observation stage skipped: set RUN_GEORGE_OBSERVATION_STAGE=True after configuring the raw data bundle.


## How confidence is used

Confidence is not a final label. It controls the strength and admissibility of the reconstructed path:

- high-confidence direct/video/flight evidence can contribute speed, displacement, direction, and arrival geometry;
- uncertainty is retained when the path is weak, so the resolver can prefer `NO_SWITCH` or `CONTESTED`;
- `p_dead` can support dead-ball reasoning;
- `p_carrier` is provenance for the HMM reconstruction and is not independently added as a second ownership vote.

The transition rule remains: a reconstructed path can propose a contact or receiver, but a switch needs contact/motion evidence and persistent post-touch team control.

In [4]:
if 'features' in locals():
    audit_columns = [c for c in [
        'ball_fill_source', 'ball_fill_source_class',
        'ball_fill_observation_confidence', 'ball_fill_sigma_m',
        'ball_fill_p_video', 'ball_fill_p_carrier', 'ball_fill_p_dead',
        'ball_fill_speed_mps', 'ball_fill_direction_change_rad',
        'scenario_context', 'anchor_type', 'missing_ball_cause'
    ] if c in features.columns]
    display(features[audit_columns].head())
else:
    print('Run the observation stage first to inspect downstream confidence/evidence columns.')

Run the observation stage first to inspect downstream confidence/evidence columns.


## Handoff to the possession model

This notebook stops at the shared observation/feature boundary. The final labels are still generated by the canonical runner and its interval resolver. George supplies a better explanation of hidden ball motion; it does not override our contact anchors, control anchors, restart constraints, `NO_SWITCH`, `CONTESTED`, or final H/A/D/X decision.

This separation lets us evaluate whether better missing-ball observations actually improve transition decisions instead of confusing trajectory accuracy with possession accuracy.